# GAF-CNN — All Outputs of the Run

**Index of every file produced by `gaf_cnn.sh`, with download links**

<small>Notebook 3 of 3 &middot; generated automatically from the output files of the `gaf_cnn.sh` run, so it is reproducible: every number below is read from the files the run produced.</small>

## 0. Setup — imports and paths

The cell below finds the `GAF_CNN_Complete` folder automatically, so the notebook works wherever the folder is placed.

In [1]:
import json, os, re, sys, glob, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink, Image as IPImage

%matplotlib inline


def find_bundle_root(start=None):
    """Walk up until we find the folder that contains run_output/gaf_cnn_sh."""
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "run_output" / "gaf_cnn_sh" / "training_console.txt").exists():
            return cand
        if (cand / "training_console.txt").exists():      # notebook inside run_output
            return cand.parent.parent
    raise FileNotFoundError("Could not find run_output/gaf_cnn_sh - run the notebook from inside "
                            "the GAF_CNN_Complete folder")


BUNDLE = find_bundle_root()
RUN    = BUNDLE / "run_output" / "gaf_cnn_sh"
CODE   = BUNDLE / "code"
NB_DIR = BUNDLE / "notebooks"
DATA   = BUNDLE / "data"
sys.path.insert(0, str(CODE))

# keep line length sane in the tables
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("bundle folder :", BUNDLE)
print("run folder    :", RUN)
print("files in run  :", sorted(p.name for p in RUN.iterdir()))

bundle folder : /home/user/CHEM_GAF_CNN/GAF_CNN_Complete
run folder    : /home/user/CHEM_GAF_CNN/GAF_CNN_Complete/run_output/gaf_cnn_sh
files in run  : ['model', 'test_eval', 'training_console.txt', 'training_console_clean.txt', 'training_curve_1_1.png', 'training_curve_1_2.png', 'training_epochs.csv', 'training_history.json']


## 1. The run

`gaf_cnn.sh` = training + test of the repository's CNN. It produced the outputs listed below,
each one readable in its own notebook:

| # | Notebook | Covers |
|---|---|---|
| 01 | `GAF_CNN_Run_01_Training_Output.ipynb` | training log, curves, epoch table, trained model |
| 02 | `GAF_CNN_Run_02_Test_Output.ipynb` | predictions, confusion matrix, accuracy, classification report |
| 03 | this notebook | index of all output files + download links |

## 2. Every output file

In [2]:
rows = []
for p in sorted(RUN.rglob("*")):
    if p.is_file():
        rows.append({
            "output": p.relative_to(RUN).as_posix(),
            "what it is": {
                "training_console.txt": "full console output of gaf_cnn.sh",
                "training_history.json": "accuracy / loss of every epoch",
                "training_epochs.csv": "the same history as a table",
                "training_curve_": "figure plotted by the script (--plot)",
            }.get(p.name, "test / evaluation output" if "test_eval" in p.parts else ""),
            "size": f"{p.stat().st_size:,} B",
        })
outputs = pd.DataFrame(rows)
display(outputs)

display(Markdown(f"**{len(outputs)} output files**, "
                 f"{sum(p.stat().st_size for p in RUN.rglob('*') if p.is_file()):,} bytes in total."))

,output,what it is,size
0,model/CNN_5classes__epo_4000_pat_1000_batch_16...,,"230,098 B"
1,test_eval/shuffled_classification_report.txt,test / evaluation output,798 B
2,test_eval/shuffled_confusion_matrix.csv,test / evaluation output,193 B
3,test_eval/shuffled_confusion_matrix.png,test / evaluation output,"60,978 B"
4,test_eval/shuffled_full_output.txt,test / evaluation output,"15,121 B"
5,test_eval/shuffled_predictions.csv,test / evaluation output,"3,426 B"
6,test_eval/shuffled_summary.json,test / evaluation output,"1,046 B"
7,training_console.txt,full console output of gaf_cnn.sh,"2,295,298 B"
8,training_console_clean.txt,,"1,403,187 B"
9,training_curve_1_1.png,,"50,644 B"


**13 output files**, 4,528,919 bytes in total.

## 3. Accuracy of this run, next to the repository's original model

This run trained a fresh model. The table compares its test accuracy with the model that was already in the repository, on the same test images.

In [3]:
import importlib
import tensorflow as tf
import gaf_cnn_full_output as lib
from gaf_cnn import simple_cnn
%matplotlib inline

model_path = (sorted(RUN.glob("model/*.keras")) or sorted(CODE.glob("CNN_*classes_*.keras")))[0]


def accuracy_of(path, test_dir):
    """Same evaluation as in notebook 02, only the headline numbers."""
    import contextlib, io as _io
    model = simple_cnn(n_classes=5)
    model(tf.zeros((1, 224, 224, 3)))
    model.load_weights(path)
    with contextlib.redirect_stdout(_io.StringIO()):
        s = lib.run_split(model, str(test_dir), "compare", str(RUN / "compare_tmp"), str(path))
    return s["accuracy_percent"], s["n_correct"], s["n_test_images"]


orig = BUNDLE / "original_model" / "CNN_5classes__epo_4000_pat_1000_batch_16_tl.keras"
test_dir = DATA / "gafimages" / "shuffled" / "Test"
rows = []
new_acc, new_ok, n = accuracy_of(model_path, test_dir)
rows.append({"model": f"this run ({model_path.name})", "test split": "shuffled",
             "correct": f"{new_ok}/{n}", "accuracy %": round(new_acc, 2)})
if orig.exists():
    o_acc, o_ok, _ = accuracy_of(orig, test_dir)
    rows.append({"model": "original model of the repository", "test split": "shuffled",
                 "correct": f"{o_ok}/{n}", "accuracy %": round(o_acc, 2)})
display(pd.DataFrame(rows))

/home/user/.venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


/home/user/.venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:415: UserWarning: Skipping variable loading for optimizer 'SGD', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


/home/user/.venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


/home/user/.venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/user/.venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:415: UserWarning: Skipping variable loading for optimizer 'SGD', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/home/user/.venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_cal

,model,test split,correct,accuracy %
0,this run (CNN_5classes__epo_4000_pat_1000_batc...,shuffled,23/25,92.0
1,original model of the repository,shuffled,24/25,96.0


## 4. Download everything

In [4]:
import zipfile
ZIP = BUNDLE / "GAF_CNN_Run_Outputs.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN.rglob("*")):
        if p.is_file() and p.name != "compare_tmp":
            z.write(p, arcname=p.relative_to(BUNDLE).as_posix())
    for nb in sorted(NB_DIR.glob("*.ipynb")):
        z.write(nb, arcname=f"notebooks/{nb.name}")
print(f"{ZIP.name}: {ZIP.stat().st_size:,} bytes, "
      f"{len(zipfile.ZipFile(ZIP).namelist())} files")

display(Markdown("### Download links"))
for label, p in [
    ("All run outputs (ZIP)", ZIP),
    ("Full console output of gaf_cnn.sh", RUN / "training_console.txt"),
    ("Trained model produced by this run", model_path),
    ("Training notebook", NB_DIR / "GAF_CNN_Run_01_Training_Output.ipynb"),
    ("Test notebook", NB_DIR / "GAF_CNN_Run_02_Test_Output.ipynb"),
    ("Raw results (CSV) of the test", RUN / "test_eval" / "shuffled_predictions.csv"),
]:
    if Path(p).exists():
        display(FileLink(str(p), result_html_prefix=f"{label}: "))
    else:
        print(f"{label}: not found")

GAF_CNN_Run_Outputs.zip: 1,167,608 bytes, 23 files


### Download links

/home/user/CHEM_GAF_CNN/GAF_CNN_Complete/GAF_CNN_Run_Outputs.zip

/home/user/CHEM_GAF_CNN/GAF_CNN_Complete/run_output/gaf_cnn_sh/training_console.txt

/home/user/CHEM_GAF_CNN/GAF_CNN_Complete/run_output/gaf_cnn_sh/model/CNN_5classes__epo_4000_pat_1000_batch_16_tl.keras

/home/user/CHEM_GAF_CNN/GAF_CNN_Complete/notebooks/GAF_CNN_Run_01_Training_Output.ipynb

/home/user/CHEM_GAF_CNN/GAF_CNN_Complete/notebooks/GAF_CNN_Run_02_Test_Output.ipynb

/home/user/CHEM_GAF_CNN/GAF_CNN_Complete/run_output/gaf_cnn_sh/test_eval/shuffled_predictions.csv

---
### Summary

`gaf_cnn.sh` was run end to end on the repository's own data (`data/gafimages/shuffled`).
Training and test outputs are stored in `run_output/gaf_cnn_sh/` and each of them is readable in
these three notebooks. All numbers come from the files the run produced, so the notebooks are
reproducible without retraining.
